In [1]:
#!/usr/bin/env python3
import rospy
from actionlib import SimpleActionClient
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PlanningFeedback
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Pose, Twist
from std_msgs.msg import Float32
from sensor_msgs.msg import LaserScan
import ipywidgets as widgets
from ipywidgets import Button, Layout, HBox, VBox
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import random
import yaml
from gazebo_msgs.srv import SetModelState

from gazebo_msgs.msg import ModelState



class TargetManager:
    def __init__(self):
        # Variabili per la gestione dei target e dei grafici
        self.target_given_list = []
        self.target_reached_list = []
        self.target_cancelled_list = []
        self.target_random_list = None
        self.target_times = {}  # used to store times of targets
        self.times_recap = widgets.Output()
        self.num_targ =0

        # processing data
        self.pose = Pose()
        self.goal = PlanningGoal()
        self.stat = ""
        self.vel = Twist()
        self.xdata, self.ydata = [], []
        self.curr_x = 0.0
        self.curr_y = 0.0
        self.clos_obstac = 0.0
        self.running = False
        
        

        self.lab_feed = widgets.Label(value = "")

        # Box to see the list of targets
        self.given_list_box = widgets.VBox([])
        self.reached_list_box = widgets.VBox([])
        self.cancelled_list_box = widgets.VBox([])
        
        # widgets buttons and texts
        self.start_b = None
        self.cancel_b = None
        self.rand_start_b = None
        self.x_input = None
        self.y_input = None
        self.text_x = None
        self.text_y = None
        self.text_ob = None
        
        # floag to avoid to miss some targets
        self.target_completed = False  
        self.done_targets =False
        
        # debug
        self.feed = widgets.Label(value ="feed:")
        self.sono_qui =widgets.Label(value ="")
        
        # Definizione delle callback per ROS
        self.client = SimpleActionClient("reaching_goal", PlanningAction)
        self.client.wait_for_server()
        
        #ubscrbers and publishers
        rospy.Subscriber("/odom", Odometry, self.pos_vel_callback)
        rospy.Subscriber("/reaching_goal/feedback", PlanningFeedback, self.feedback_callback)
        rospy.Subscriber('/scan', LaserScan, self.laser_callback)
        self.publish_distance = rospy.Publisher('closest_obstacle', Float32, queue_size=10)
        rospy.wait_for_service('/gazebo/set_model_state')
        self.set_state = rospy.ServiceProxy('/gazebo/set_model_state', SetModelState)

        # charts and graphic
        %matplotlib widget
        plt.ioff()
        self.fig, (self.ax2, self.ax1) = plt.subplots(1, 2, figsize=(9, 5))
        self.fig.patch.set_visible(False)
        self.ax1.set_title(
                            f'Reached vs Cancelled\nsent: {len(self.target_given_list)}, '
                            f'reached: {len(self.target_reached_list)}, cancelled: {len(self.target_cancelled_list)}',
                            fontsize=10
                        )
        self.ax2.set_title("Live trajectory")
        self.ax2.grid()
        self.ln, = self.ax2.plot([], [], 'r-')
        self.ax2.set_xlim(10, -10)
        self.ax2.set_ylim(10, -10)
        plt.ion()
        self.ani = FuncAnimation(self.fig, self.update_plots, interval=100)
        with open("results_target1.yaml","w") as f:
            pass                
        
    def pos_vel_callback(self, msg):
        # obtain the position and publishes it
        self.vel = msg.twist.twist
        self.xdata.append(msg.pose.pose.position.x)
        self.ydata.append(msg.pose.pose.position.y)
        self.curr_x = msg.pose.pose.position.x
        self.curr_y = msg.pose.pose.position.y

        
    def feedback_callback(self, feedback_: PlanningFeedback) -> None:
        self.stat = feedback_.feedback.stat
        self.process_feedback()

    def laser_callback(self, msg):
        # obtain the obstacles and compute the min distance
        min_distance = min(msg.ranges)
        closest_obstacle_msg = Float32()
        closest_obstacle_msg.data = min_distance
        self.publish_distance.publish(closest_obstacle_msg)
        self.clos_obstac = min_distance

    def process_feedback(self):
        self.feed.value = self.stat
        # when target is completed and I am sure I don't process the same target 
        if self.stat == "Target reached!" and not self.target_completed: 
            self.target_completed = True  
            self.sono_qui.value = "1. qui"
            # obtain target from the list targets list and move it into the list of reached targets
            current_target = self.target_random_list.pop(0)  
            self.target_reached_list.append(current_target)  
            # update the widget , showing in green the reached target
            self.update_target_list("reached")

            # preparing the data to register: target, duration, end and start 
            target_str = str({'aim_x': current_target[0], 'aim_y': current_target[1]})
            end_time = rospy.Time.now()
            target_id = str(self.num_targ)  # get the current target ID as string

            
            # if the target reached is present inside the dictionary ( just a check), update rest of the fields of the dictionary
            if target_id in self.target_times:
                self.sono_qui.value = f'2. Settato tempi, vettore: {self.target_random_list}'

                #  Record end time and calculate duration
                self.target_times[target_id]['end'] = end_time
                duration = end_time - self.target_times[target_id]['start']
                self.target_times[target_id]['duration'] = duration
                rospy.loginfo(f"Target ID {target_id} completato in {duration.to_sec()} secondi.")

                # Prepare data to save (converted to seconds)
                times = self.target_times[str(target_id)]
                data_to_save = {
                    'id' :times['id'],
                    'target' :times['target'],
                    'start': times['start'].to_sec() if times['start'] else None,
                    'end': times['end'].to_sec() if times['end'] else None,
                    'duration': times['duration'].to_sec() if times['duration'] else None
                }
                
                # Write to YAML file (append mode)
                with open("results_target1.yaml", "a") as f:
                    yaml.dump({f"target_{target_id}": data_to_save}, f, explicit_start=True)


            # if after having obtained the target from the list I see the list is still not empty then I set as new target the next element 
            if len(self.target_random_list) > 0:
                self.sono_qui.value = f'3. nuovo target, vettore: {self.target_random_list}' 
                print("Prossimo target")
                next_target = self.target_random_list[0]
                self.teleport()
                self.do_goal(next_target[0], next_target[1])
                self.target_completed = False  
            else:
                self.sono_qui.value = f'4. elinimo targets, vettore: {self.target_random_list}'
                print("Fine dei target")
                
                self.del_goal()
                self.target_random_list = []
                self.cancel_b.disabled = True
                self.start_b.disabled = False
                self.rand_start_b.disabled = False
                self.target_completed = False 
                self.update_times_recap()
                self.teleport()
                self.done_targets =True
                
        elif self.stat == "Target cancelled!":
#                self.target_completed = False 
                self.del_goal()
                self.cancel_b.disabled = True
                self.start_b.disabled = False
                self.rand_start_b.disabled = False
                current_target = self.target_random_list.pop(0)  
                self.target_cancelled_list.append(current_target) 
                self.update_target_list("cancelled")
                self.teleport()
                self.num_targ =0
                self.target_random_list = []
                self.done_targets =True

                
    def update_times_recap(self):
        # gives me a recap , can be seen in yaml file
        with self.times_recap:
            self.times_recap.clear_output()
            for target_str, times in self.target_times.items():
                start = times['start'].to_sec() if times['start'] else None
                end = times['end'].to_sec() if times['end'] else None
                duration = times['duration'].to_sec() if times['duration'] else None
                print(f"Target: {target_str}")
                print(f"Inizio:   {start}")
                print(f"Fine:     {end}")
                print(f"Durata:  {duration} s\n")
    
    def teleport(self):
        # teleporting the robot in (0,1)
        try:
            state_msg = ModelState()
            state_msg.model_name = 'robot1' 
            state_msg.pose.position.x = 0.0
            state_msg.pose.position.y = 1.0
            state_msg.pose.position.z = 0.0
            state_msg.pose.orientation.x = 0.0
            state_msg.pose.orientation.y = 0.0
            state_msg.pose.orientation.z = 0.0
            state_msg.pose.orientation.w = 0.0 

            state_msg.twist.linear.x = 0.0
            state_msg.twist.linear.y = 0.0
            state_msg.twist.linear.z = 0.0
            state_msg.twist.angular.x = 0.0
            state_msg.twist.angular.y = 0.0
            state_msg.twist.angular.z = 0.0

            state_msg.reference_frame = 'world'

            resp = self.set_state(state_msg)
            rospy.loginfo(f"Success: {resp.success}, Status: {resp.status_message}")

        except rospy.ServiceException as e:
            rospy.logerr(f"Service call failed: {e}")
 
    def update_target_list(self, status):
#         self.sono_qui.value = f'5. stat : {status}'

        # if reaches I change the color of the target in green
        if status == "reached":
            new_target_rea = widgets.HTML(f'<span style="color: black;">{self.target_reached_list[-1]}</span>')
            self.reached_list_box.children += (new_target_rea,)
            new_target_can = widgets.HTML(f'<br>')
            self.cancelled_list_box.children += (new_target_can,)
            self.given_list_box.children[-1].value = f'<span style="color: green;">{self.target_given_list[-1]}</span>'
        
        # if cancel I change the color of the target in red
        elif status == "cancelled":
            new_target_rea = widgets.HTML(f'<br>')
            self.reached_list_box.children += (new_target_rea,)
            new_target_can = widgets.HTML(f'<span style="color: black;">{self.target_cancelled_list[-1]}</span>')
            self.cancelled_list_box.children += (new_target_can,)
            self.given_list_box.children[-1].value = f'<span style="color: red;">{self.target_given_list[-1]}</span>'
            
            
            
    def do_goal(self, x, y):
        self.done_targets = False
        # building the target with the element given 
        self.pose.position.x = x
        self.pose.position.y = y
        target = {'aim_x': self.pose.position.x, 'aim_y': self.pose.position.y}
        
        # adding the target to the target list 
        self.target_given_list.append(target)
        
        #updating the widget showing th current target
        new_target = widgets.HTML(f'<span style="color: black;">{target}</span>')
        self.given_list_box.children += (new_target,)
        
        # building and sending the goal 
        self.goal.target_pose.pose = self.pose
        self.client.send_goal(self.goal)
        
        # storing the initial starting moment 
        start_time = rospy.Time.now()
        self.num_targ +=1
        
        # storing the data in the dictionary to associate to each target start, end and duration time
        self.target_times[str(self.num_targ)] = {'id':self.num_targ,'target':target, 'start': start_time, 'end': None, 'duration': None}

    def del_goal(self):
        self.client.cancel_all_goals()
        
    def update_plots(self, frame):
        # updating pie chart and the map
        if len(self.target_reached_list) != 0 or len(self.target_cancelled_list) != 0:
            self.ax1.clear()
            sizes = [len(self.target_reached_list), len(self.target_cancelled_list)]
            labels = ['Reached', 'Cancelled']
            self.ax1.pie(sizes, labels=labels, autopct='%1.1f%%')
            self.ax1.set_title(f'Reached vs Cancelled \n sent: {len(self.target_given_list)}, reached: {len(self.target_reached_list)}, cancelled: {len(self.target_cancelled_list)}', fontsize=10)
        
        self.ln.set_data(self.xdata, self.ydata)
        return self.ln,


In [2]:
rospy.init_node("target_client")

# Inizializzazione della classe TargetManager
target_manager = TargetManager()
    

def on_start_button_clicked( b):
    start_b.disabled = True
    cancel_b.disabled = False
    target_manager.do_goal(x_input.value, y_input.value)
    
    
def on_start_rand_button_clicked( b):
#     print("qui")
#     print(f'lungh vettore  {len(target_manager.target_random_list)}, cond :{target_manager.done_targets}')
    # if cancelled or the targets have already been reached, then i cannot play "random start" unless I give other targets
    if target_manager.done_targets == True and len(target_manager.target_random_list) == 0:
        return
    start_b.disabled = True
    cancel_b.disabled = False
    targ = target_manager.target_random_list[0]
    target_manager.do_goal(targ[0],targ[1])

def on_cancel_button_clicked(b):
    cancel_b.disabled = True
    start_b.disabled =False
    x_input.disabled = False
    y_input.disabled = False
    x_input.value = 0.0
    y_input.value = 0.0
    target_manager.del_goal()
    
def update_widgets( event):
    text_x.value = f'{target_manager.curr_x:.4f}'
    text_y.value = f'{target_manager.curr_y:.4f}'
    text_ob.value = f'{target_manager.clos_obstac:.4f}'


In [3]:
# Input per il target
message_label = widgets.HTML(
    value='<h2 style="margin:0px; padding-left:0px;">Set target (x,y):</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
)
x_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='80px'))
y_input = widgets.FloatText(value=0.0, layout=widgets.Layout(width='80px'))
box_input = HBox([message_label, x_input, y_input], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))

# declaring number of experiments
num_departures = 3
random_positions = [(random.randint(-9, 9), random.randint(2, 3)) for _ in range(num_departures)]


# start and cancel buttons
start_b = widgets.Button(description=' Start', layout=Layout(width='50%', height='80px'), disabled=False, button_style='success')
start_rand_b = widgets.Button(description=' Start random', layout=Layout(width='50%', height='80px'), disabled=False, button_style='success')

cancel_b = widgets.Button(description='Cancel target', layout=Layout(width='50%', height='80px'), disabled=True, button_style='danger')

# initialization of the manager target
target_manager.start_b = start_b
target_manager.cancel_b = cancel_b
target_manager.x_input = x_input
target_manager.y_input = y_input
target_manager.rand_start_b = start_rand_b
target_manager.target_random_list =random_positions
print(random_positions)

# texts for data of the closest obstacle and position 
text_x = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))
text_y = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))
text_ob = widgets.Text(value="0.0", disabled=True, layout=widgets.Layout(width='80px'))

rospy.Timer(rospy.Duration(0.1), update_widgets)
# rospy.Timer(rospy.Duration(0.1), update_toggle)


# texts of the closest obstacle and position 
label_pos = widgets.HTML(value='<h2 style="margin:0px; padding-right:10px;">Position (x,y):</h2>', layout=widgets.Layout(display='flex', align_items='center'))
label_ob = widgets.HTML(value='<h2 style="margin:0px; padding-right:10px;">Closest obstacle:</h2>', layout=widgets.Layout(display='flex', align_items='center'))
label_metres = widgets.HTML(value='<h2 style="margin:0px; padding-left:10px;">metres</h2>', layout=widgets.Layout(display='flex', align_items='center'))

# wdget of the closest obstacle and position 
box_pos = HBox([label_pos, text_x, text_y], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))
box_ob = HBox([label_ob, text_ob, label_metres], layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%'))

# binding buttons to functions
start_b.on_click(on_start_button_clicked)
cancel_b.on_click(on_cancel_button_clicked)
start_rand_b.on_click(on_start_rand_button_clicked)



display(box_input)
display(HBox([start_b, cancel_b]))
display(HBox([box_pos, box_ob]))
display(start_rand_b)
plt.show()

box_layout = Layout(border='1px solid black', width='33%', align_items='center', padding='10px')
display(HBox([
    VBox([widgets.HTML(f'<b> Target cancelled </b>'), target_manager.cancelled_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target given </b>'), target_manager.given_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target reached </b>'), target_manager.reached_list_box], layout=box_layout)
], layout=Layout(width='100%')))

# display(widgets.VBox([
#     target_manager.times_recap  # mostra il riepilogo dei tempi
# ]))

display(target_manager.feed)
display(target_manager.sono_qui)


[(-9, 3), (-5, 2), (7, 3)]


Button(button_style='success', description=' Start random', layout=Layout(height='80px', width='50%'), style=B…

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Label(value='feed:')

Label(value='')

In [15]:
import numpy as np
import matplotlib.pyplot as plt



def get_random(num_departures):
    count = 0
    random_vector = []
    while count != num_departures:
        num_x = random.randint(-9, 9)
        num_y = random.randint(-9, 9)
        rand_pos = (num_x, num_y)
        print(rand_pos)
        if (verify_target(rand_pos) == True):
            random_vector.append(rand_pos)
            count +=1
    return random_vector
    
def verify_target(rand_pos):
    x_val,y_val=  rand_pos
    if (-9 <= x_val <= 9) and (-9 <= y_input.value <= 9) and not (
                (abs(x_val) == 9 and abs(y_val) == 9) or
                (abs(x_val) == 8 and abs(y_val) == 9) or
                (abs(x_val) == 9 and abs(y_val) == 8) or
                (abs(x_val) == 8 and abs(y_val) == 8)):
        return True
    print("aglia")
    return False
            





num_departures = 30
random_positions = get_random(num_departures)
print(random_positions)


# fig, ax = plt.subplots()
# ax.hist(random_positions, bins=30, density=True, alpha=0.6, color='b', edgecolor='black')
# ax.set_title('Istogramma dei Dati (Truncated Normal)')
# ax.set_xlabel('Valori')
# ax.set_ylabel('Densità')
# plt.show()


(1, -5)
(4, -9)
(8, 4)
(8, 9)
aglia
(0, 0)
(5, 9)
(-2, -3)
(-7, -9)
(-3, -1)
(2, -2)
(-1, -2)
(5, 1)
(6, -4)
(9, -3)
(-9, 1)
(-1, 9)
(-6, -2)
(6, -6)
(8, -6)
(-5, -2)
(4, -6)
(-5, -2)
(3, -2)
(7, -7)
(7, -3)
(0, -5)
(-4, 7)
(4, -9)
(-8, 6)
(7, -9)
(-9, -4)
[(1, -5), (4, -9), (8, 4), (0, 0), (5, 9), (-2, -3), (-7, -9), (-3, -1), (2, -2), (-1, -2), (5, 1), (6, -4), (9, -3), (-9, 1), (-1, 9), (-6, -2), (6, -6), (8, -6), (-5, -2), (4, -6), (-5, -2), (3, -2), (7, -7), (7, -3), (0, -5), (-4, 7), (4, -9), (-8, 6), (7, -9), (-9, -4)]


In [14]:
import statistics

vett_duration = []

for entry in target_manager.target_times.values():
    if entry['duration'] is not None:
        vett_duration.append(entry['duration'].to_sec())

if vett_duration:
    mean = statistics.mean(vett_duration)
    print("Media durate:", mean)
    dev_std = statistics.stdev(vett_duration)
    print("Deviazione standard:", dev_std)
else:
    print("Nessuna durata registrata.")


Nessuna durata registrata.


In [11]:
print(random_positions)

[(9, -4), (-4, -4), (-1, -7), (-8, 0), (6, -6), (3, 0), (8, -5), (3, -7), (8, 5), (-6, -5), (-8, -7), (1, -4), (5, 9), (4, 2), (6, -2), (-2, 2), (3, 1), (1, 5), (3, -2), (0, 0), (8, -5), (-7, -8), (1, 6), (-9, -4), (5, -4), (2, 7), (7, 1), (-9, -1), (9, -2), (5, 9)]
